Data augmentation is a technique where we increase our dataset by adding different transformations to our data to both increase our dataset and imporve our models robustness (Decrease chances of overfitting). The 5 techniques that ill be using to imporve my model will be: Random Rotation, Random Brightness, Random Shifts, Random Flips, Random Zoom. 
To better explain these transformations: 
Random Rotations: You define a max rotation and the api will pick any random rotation given the defined range. 
Random Brightness: Basically the same thing, you define your range of brightness and the api will pick randomly given that defined range. 
Random Shifts: Shift the image’s content in the horizontal and vertical direction by a specified range of translation.
Random Flips: Set the “horizontal_flip” and “vertical_flip” arguments to ‘True”, and the API will randomly flip the image.
Random Zoom: Specify a range for zoom out (min) and zoom in (max) using the “zoom_range” argument, and the API will randomly pick a value and give us a transformed image.

Why does training on a limited dataset lead to overfitting, and how does augmentation help address this? 
Having a limited dataset means that the model will be going over the same image multiple times leading to memorization of said patterns, this leads to overfitting. When we apply data augmentations techniques the dataset becomes well versed and larger, this makes the model instead of memorize the patterns it instead learns the patterns making the model better on real world data. 

What types of transformations are most commonly applied to image data, and when is each appropriate?
Ive mentioned the transformations already above, now to answer the when is each approppiate question. 
Random Rotations: Used when the orientation of the object doesnt matter common cases are: Vegetation, roads, sattelites and medical images. (Not so useful when it comes to identifying digits)
Random Brightness: Used when lighting conditions vary in real life. Common cases are: outdoor images, satellite imagery, shadows, clouds. (Not so useful when pixel intensity directly represents meaning)
Random Shifts: Used when object location doesn’t matter. (Not so useful when the location of the object matters)
Random Flips: Used when Scene is symmetric / orientation doesn’t matter. (satellite images, landscapes) (Not so useful in text, traffic signs, human faces sometimes)
Random Zoom: Used when Objects appear at different scales in real life. (The downside of this is that zooming removes important context or crops out key features.)

How does augmentation interact with label structure, particularly for pixel level prediction tasks?
Well the main take away I got was that if you do a data augmentation transformation on a sample you MUST do it aswell on the label. If not the model will be confused and not perform to standard.

In [ ]:
import os
import numpy as np
import rasterio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torch.optim as optim

from torchvision import transforms
import torchvision.transforms.functional as TF
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight


In [2]:
samples = sorted(os.listdir("samples"))
labels = sorted(os.listdir("labels"))

print("SAMPLES:")
for i in range(5):
    print(samples[i])

print("\nLABELS:")
for i in range(5):
    print(labels[i])

SAMPLES:
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_0.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_1.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_10.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_100.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_101.tiff

LABELS:
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_0.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_1.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_10.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_100.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_101.tiff


In [3]:
# We need to normalize our "ndvi values" since initially our values were just the pixel value. 
# Once weve normalized it our values will be -1 to 1. 
# Once we have our actual NDVI values well make out 2 classes 0 for no vegetation aka ndvi values less than 0.3 and 1 for anything above 0.3 aka vegetation. 
def normalize_ndvi(ndvi):
    ndvi = ndvi.astype(np.float32)
    ndvi = (ndvi - ndvi.min()) / (ndvi.max() - ndvi.min() + 1e-8)
    ndvi = ndvi * 2.0 - 1.0
    return ndvi

def ndvi_to_binary_class(ndvi):
    classes = np.zeros_like(ndvi, dtype=np.uint8)

    # 0 = non-vegetation
    # 1 = vegetation
    classes[ndvi >= 0.3] = 1

    return classes

In [4]:
X_list = []
y_list = []

for sample_name in samples:
    sample_path = os.path.join("samples", sample_name)

    label_name = sample_name.replace("img", "ndvi")
    label_path = os.path.join("labels", label_name)

    if not os.path.exists(label_path):
        print(f"Missing label for {sample_name}")
        continue

    with rasterio.open(sample_path) as src: # Loading our sample
        img = src.read().astype(np.float32)

    with rasterio.open(label_path) as src: # Loading our label 
        lbl = src.read(1)

    ndvi_norm = normalize_ndvi(lbl) # Utilizing our already defined function to normalize each label.
    class_lbl = ndvi_to_binary_class(ndvi_norm) # Forming our classes based on there ndvi values.

    X_list.append(img) 
    y_list.append(class_lbl)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.uint8)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique classes in y:", np.unique(y))

c:\Users\dr274\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rasterio\__init__.py:379: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


X shape: (614, 3, 256, 256)
y shape: (614, 256, 256)
Unique classes in y: [0 1]


In [5]:
# Dividing our data set into training data and testing data (Make sure its not memorizing our data and actually learning)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (491, 3, 256, 256)
X_test: (123, 3, 256, 256)
y_train: (491, 256, 256)
y_test: (123, 256, 256)


In [ ]:
# Turning our dataset into a dataset of tensors. 
# Define a way our model can actually access these tensors. 
class VegetationDataset(Dataset):
    def __init__(self, images, masks, augment=False):
        self.images = images
        self.masks = masks
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]   
        mask = self.masks[idx]     

        image = torch.tensor(image, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)

        # Normalize image values if needed
        image = image / 255.0

        if self.augment:
            # Random horizontal flip
            if random.random() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask.unsqueeze(0)).squeeze(0)

            # Random vertical flip
            if random.random() > 0.5:
                image = TF.vflip(image)
                mask = TF.vflip(mask.unsqueeze(0)).squeeze(0)

            # Random rotation
            angle = random.uniform(-30, 30)
            image = TF.rotate(image, angle)
            mask = TF.rotate(mask.unsqueeze(0), angle).squeeze(0)

            # Brightness / contrast ONLY for image
            brightness_factor = random.uniform(0.9, 1.1)
            contrast_factor = random.uniform(0.9, 1.1)
            image = TF.adjust_brightness(image, brightness_factor)
            image = TF.adjust_contrast(image, contrast_factor)

        return image, mask

In [7]:
train_dataset = VegetationDataset(X_train, y_train, augment=True)
test_dataset = VegetationDataset(X_test, y_test, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [ ]:
# Built a Simple CNN with 8 layers, we used an encoder - decoder cnn since we were dealing with pixels. The encoder allowed us to recognize the patterns of the pixels more easily and then 
# We utilized a decoder to upsample our image (Aka go back to original size after our encoder shrank it). The decoder allowed us to restore the pixel details
class SimpleSegCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(SimpleSegCNN, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, num_classes, kernel_size=1)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
# # Regularization
# class SimpleSegCNN(nn.Module):
#     def __init__(self, num_classes=2):
#         super(SimpleSegCNN, self).__init__()

#         self.encoder = nn.Sequential(
#             nn.Conv2d(3, 16, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.Dropout2d(p=0.3),

#             nn.Conv2d(16, 32, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.Dropout2d(p=0.3),

#             nn.MaxPool2d(2),

#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.Dropout2d(p=0.3),

#             nn.Conv2d(64, 64, kernel_size=3, padding=1),
#             nn.ReLU()
#         )

#         self.decoder = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

#             nn.Conv2d(64, 32, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.Dropout2d(p=0.3),

#             nn.Conv2d(32, num_classes, kernel_size=1)
#         )

#     def forward(self, x):
#         x = self.encoder(x)
#         x = self.decoder(x)
#         return x

In [ ]:
#  # Extra Layer
# class SimpleSegCNN(nn.Module):
#     def __init__(self, num_classes=2):
#         super(SimpleSegCNN, self).__init__()

#         self.encoder = nn.Sequential(
#             nn.Conv2d(3, 16, kernel_size=3, padding=1),
#             nn.ReLU(),

#             nn.Conv2d(16, 32, kernel_size=3, padding=1),
#             nn.ReLU(),

#             nn.MaxPool2d(2),

#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.ReLU(),

#             nn.Conv2d(64, 64, kernel_size=3, padding=1),
#             nn.ReLU(),

#             # NEW LAYER
#             nn.Conv2d(64, 128, kernel_size=3, padding=1),
#             nn.ReLU()
#         )

#         self.decoder = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

#             nn.Conv2d(128, 64, kernel_size=3, padding=1),
#             nn.ReLU(),

#             nn.Conv2d(64, num_classes, kernel_size=1)
#         )

#     def forward(self, x):
#         x = self.encoder(x)
#         x = self.decoder(x)
#         return x

In [ ]:
# Sigmoid
#  class SimpleSegCNN(nn.Module):
#     def __init__(self, num_classes=2):
#         super(SimpleSegCNN, self).__init__()

#         self.encoder = nn.Sequential(
#             nn.Conv2d(3, 16, kernel_size=3, padding=1),
#             nn.Sigmoid(),

#             nn.Conv2d(16, 32, kernel_size=3, padding=1),
#             nn.Sigmoid(),

#             nn.MaxPool2d(2),

#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.Sigmoid(),

#             nn.Conv2d(64, 64, kernel_size=3, padding=1),
#             nn.Sigmoid()
#         )

#         self.decoder = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

#             nn.Conv2d(64, 32, kernel_size=3, padding=1),
#             nn.Sigmoid(),

#             nn.Conv2d(32, num_classes, kernel_size=1)
#         )

#     def forward(self, x):
#         x = self.encoder(x)
#         x = self.decoder(x)
#         return x

In [ ]:
# Less layers: 
#  class SimpleSegCNN(nn.Module):
#     def __init__(self, num_classes=2):
#         super(SimpleSegCNN, self).__init__()

#         self.encoder = nn.Sequential(
#             nn.Conv2d(3, 16, kernel_size=3, padding=1),
#             nn.ReLU(),

#             nn.MaxPool2d(2),

#             nn.Conv2d(16, 32, kernel_size=3, padding=1),
#             nn.ReLU()
#         )

#         self.decoder = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

#             nn.Conv2d(32, num_classes, kernel_size=1)
#         )

#     def forward(self, x):
#         x = self.encoder(x)
#         x = self.decoder(x)
#         return x

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = SimpleSegCNN(num_classes=2).to(device)

Loss_Function = nn.CrossEntropyLoss()
# optimizer = torch.optim.RMSprop(
#     model.parameters(),
#     lr=0.001
# )
# optimizer = torch.optim.SGD(
#     model.parameters(),
#     lr=0.01,
#     momentum=0.9
# )
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: cuda


In [ ]:
# In this Cell what we do is compute the weights by using a predefined function from sckit learn, but we first flatten our labels since we need to give them a 1D list of all the labels
# Then we set out our lost function utilizing those weights and we also make our optimizer to update our models parameters.

model = SimpleSegCNN(num_classes=2).to(device)

y_flat = y_train.reshape(-1)
classes = np.unique(y_flat)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_flat
)

weights = torch.tensor(weights, dtype=torch.float32).to(device)
print("Class weights:", weights.cpu().numpy())

Loss_Function = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

Class weights: [1.0066942 0.9933942]
SimpleSegCNN(
  (encoder): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
  )
  (decoder): Sequential(
    (0): Upsample(scale_factor=2.0, mode='bilinear')
    (1): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): ReLU()
    (3): Conv2d(32, 2, kernel_size=(1, 1), stride=(1, 1))
  )
)


In [20]:
# In this cell we train our model over 30 epochs utilizing mini batches. every prediction made the loss was calculated and displayed. 
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)   # (B, 2, H, W)
        loss = Loss_Function(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

Epoch [1/30], Loss: 0.6021
Epoch [2/30], Loss: 0.5187
Epoch [3/30], Loss: 0.4431
Epoch [4/30], Loss: 0.3697
Epoch [5/30], Loss: 0.3685
Epoch [6/30], Loss: 0.3392
Epoch [7/30], Loss: 0.3337
Epoch [8/30], Loss: 0.3172
Epoch [9/30], Loss: 0.3157
Epoch [10/30], Loss: 0.2958
Epoch [11/30], Loss: 0.2919
Epoch [12/30], Loss: 0.2786
Epoch [13/30], Loss: 0.2789
Epoch [14/30], Loss: 0.2592
Epoch [15/30], Loss: 0.2646
Epoch [16/30], Loss: 0.2850
Epoch [17/30], Loss: 0.2509
Epoch [18/30], Loss: 0.2517
Epoch [19/30], Loss: 0.2647
Epoch [20/30], Loss: 0.2567
Epoch [21/30], Loss: 0.2449
Epoch [22/30], Loss: 0.2454
Epoch [23/30], Loss: 0.2500
Epoch [24/30], Loss: 0.2451
Epoch [25/30], Loss: 0.2427
Epoch [26/30], Loss: 0.2494
Epoch [27/30], Loss: 0.2442
Epoch [28/30], Loss: 0.2415
Epoch [29/30], Loss: 0.2365
Epoch [30/30], Loss: 0.2381


In [21]:
model.eval()

all_preds = []
all_true = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.append(preds.cpu().numpy().reshape(-1))
        all_true.append(masks.cpu().numpy().reshape(-1))

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Check predictions
print("\nUnique predicted classes:", np.unique(y_pred))
print("Unique true classes:", np.unique(y_true))

# Full report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

Accuracy: 0.8897040142276422
Precision: 0.8901716484800354
Recall: 0.8897040142276422
F1-score: 0.8897345909970041

Unique predicted classes: [0 1]
Unique true classes: [0 1]

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.88      0.89   4165283
           1       0.87      0.90      0.89   3895645

    accuracy                           0.89   8060928
   macro avg       0.89      0.89      0.89   8060928
weighted avg       0.89      0.89      0.89   8060928



Experiments:
I decreased my lr from 0.001 to 0.0001 and observed a reduction on every metric by 4%. 
I increased my lr from 0.001 to 0.01 and observed a reduction on every  metric by 2%. 
Increased the Kernel Size from 3 to 5 and the padding size from 1 to 2. This showed a reduction on every metric by roughly 3%. 
When I switched the activation function from Relu to Sigmoid I observed a reduction of 25% in all 4 major evaluation metrics. 
When I switched the optimizer to the SGD with Momentum I saw a slower training speed and and saw a decline in every metric of 4%. 
When I switched the optimizer to the RMSprop I saw a slower training speed and and saw a decline in every metric of 3%. 
Applying regularization I saw a drop of roughly 22% in all metrics. 
When i added 1 more layer i saw a drop of 3% across all metrics and way longer training time. 
After Removing a convolutional block i saw a 6% decrease across all  metrics. 

(AS A BASELINE I utilized Adam optimizer and a total epoch of 30)





Comparison with CNN of Task 3: 
- Overall perfomance improved.
- There was a better class balance.
- There was a huge improvement of the recall (from 0.82 to 0.90)
- A recall trade off in class 0 from 0.92 to 0.88 
